## Контрольная работа 4 Вариант 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

### Задача 1 Численное дифференцирование (1.5 балла)

Для заданной функции $f(x)=c h(x)$ на интервале $[-1 ; 0]$ для нескольких шагов $h$
- Аналитически рассчитать вторую производную 
- Вычислить и построить аппроксимацию второй производной, используя схемы первого и второго порядка аппроксимации, построить графики вместе с истинным ответом для одного из значений $h$
- Вычислить и погрешность для каждой из аппроксимаций и построить её от шага $h$. Какие получились экспериментальные порядки аппроксимации?

In [ ]:
f = np.cosh
f2_exact = np.cosh  # f''(x) = cosh(x)

a, b = -1.0, 0.0

# --- схема первого порядка: f''(x) ≈ (f(x+h) - 2f(x) + f(x-h)) / h^2  (порядок 2?)
# Нет: схема 1-го порядка для второй производной —
# несимметричная: f''(x) ≈ (f(x) - 2f(x+h) + f(x+2h)) / h^2
# Схема 2-го порядка — центральная: (f(x+h) - 2f(x) + f(x-h)) / h^2

def d2_order1(f, x, h):
    return (f(x) - 2*f(x + h) + f(x + 2*h)) / h**2

def d2_order2(f, x, h):
    return (f(x + h) - 2*f(x) + f(x - h)) / h**2

h_demo = 0.05
x_plot = np.arange(a, b + 1e-12, h_demo)
x_plot1 = x_plot[x_plot + 2*h_demo <= b + 1e-12]

fig, ax = plt.subplots(figsize=(8, 4))
x_fine = np.linspace(a, b, 500)
ax.plot(x_fine, f2_exact(x_fine), 'k-', lw=2, label="$f''(x) = \\cosh(x)$ (точная)")
ax.plot(x_plot1, d2_order1(f, x_plot1, h_demo), 'o--', ms=4, label=f"1-й порядок, h={h_demo}")
ax.plot(x_plot, d2_order2(f, x_plot, h_demo), 's--', ms=4, label=f"2-й порядок, h={h_demo}")
ax.set_xlabel("x"); ax.set_ylabel("$f''(x)$")
ax.legend(); ax.set_title("Аппроксимация второй производной $\\cosh(x)$")
plt.tight_layout(); plt.show()

H = np.logspace(-1, -5, 30)
err1 = np.zeros_like(H)
err2 = np.zeros_like(H)

for i, h in enumerate(H):
    x_inner = np.linspace(a + 2*h, b - 2*h, 200)
    if len(x_inner) == 0:
        x_inner = np.array([(a + b) / 2])
    err1[i] = np.max(np.abs(d2_order1(f, x_inner, h) - f2_exact(x_inner)))
    err2[i] = np.max(np.abs(d2_order2(f, x_inner, h) - f2_exact(x_inner)))

p1 = np.polyfit(np.log(H), np.log(err1), 1)[0]
p2 = np.polyfit(np.log(H), np.log(err2), 1)[0]

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(H, err1, 'o-', label=f"1-й порядок (наклон {p1:.2f})")
ax.loglog(H, err2, 's-', label=f"2-й порядок (наклон {p2:.2f})")
ax.set_xlabel("h"); ax.set_ylabel("max |err|")
ax.legend(); ax.set_title("Погрешность аппроксимации $f''$ от шага $h$")
plt.tight_layout(); plt.show()

print(f"Экспериментальный порядок схемы 1: {p1:.2f}")
print(f"Экспериментальный порядок схемы 2: {p2:.2f}")

### Задача 2  Численное интегрирование (4 балла)

Вычислить приближенно интеграл 

$$
J=\int_0^{\pi / 2} \ln \sin x \cdot d x
$$


Точное значение интеграла равно $J=-1.0890045$. 

Вам нужно использовать метод, который даст не меньше 6 верных знаков после запятой. Точное значение интеграла, как и функцию `quad`, при вычислении использовать нельзя.

Подынтегральная функция $\ln\sin x$ имеет логарифмическую особенность при $x \to 0^+$:

$$\ln\sin x \approx \ln x + O(x^2)$$

Вычтем особенность: $\ln\sin x = \ln x + \underbrace{(\ln\sin x - \ln x)}_{\text{гладкая}}$, причём

$$\int_0^{\pi/2} \ln x\,dx = \frac{\pi}{2}\left(\ln\frac{\pi}{2} - 1\right)$$

Остаток $g(x) = \ln\!\frac{\sin x}{x}$ гладкий на $[0, \pi/2]$, $g(0) = 0$, поэтому его можно интегрировать составной формулой Симпсона с высокой точностью.

In [ ]:
from scipy.integrate import simpson

I_singular = (np.pi / 2) * (np.log(np.pi / 2) - 1)

def g(x):
    out = np.zeros_like(x, dtype=float)
    mask = x > 0
    out[mask] = np.log(np.sin(x[mask]) / x[mask])
    return out

J_exact = -1.0890045

npts = 1001
x = np.linspace(0, np.pi / 2, npts)
I_smooth = simpson(g(x), x=x)
J_approx = I_singular + I_smooth

print(f"I_singular  = {I_singular:.10f}")
print(f"I_smooth    = {I_smooth:.10f}")
print(f"J_approx    = {J_approx:.10f}")
print(f"J_exact     = {J_exact:.10f}")
print(f"|err|       = {abs(J_approx - J_exact):.2e}")

print("\nСходимость по числу точек:")
for n in [101, 201, 501, 1001, 2001, 5001]:
    xx = np.linspace(0, np.pi / 2, n)
    J_n = I_singular + simpson(g(xx), x=xx)
    print(f"  n = {n:>5} — J = {J_n:.10f},  |err| = {abs(J_n - J_exact):.2e}")

### Задача 3  Интерполяция (2 балла)

На отрезке $[2,5]$ для дробно-линейной функции $y(x)=\frac{x-3,5}{x+\sqrt{3}-3,5}$ построить интерполяционный полином в форме Лагранжа на сетке из __экстремумов__ полинома Чебышёва с $n$ нулями на данном отрезке. Оценить погрешность данного метода интерполяции в зависимости от $n$ - число узлов Чебышева.


In [ ]:
def y_func(x):
    return (x - 3.5) / (x + np.sqrt(3) - 3.5)

# Экстремумы полинома Чебышёва T_n(t) на [-1,1]: t_k = cos(pi*k/n), k=0,...,n
# Перенос на [a, b]: x_k = (a+b)/2 + (b-a)/2 * t_k
a_int, b_int = 2.0, 5.0

def cheb_extrema(n, a, b):
    k = np.arange(n + 1)
    t = np.cos(np.pi * k / n)
    return (a + b) / 2 + (b - a) / 2 * t

def lagrange_interp(xk, yk, x):
    n = len(xk)
    result = np.zeros_like(x, dtype=float)
    for i in range(n):
        basis = np.ones_like(x, dtype=float)
        for j in range(n):
            if j != i:
                basis *= (x - xk[j]) / (xk[i] - xk[j])
        result += yk[i] * basis
    return result

x_fine = np.linspace(a_int, b_int, 1000)
y_fine = y_func(x_fine)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for n in [5, 8, 12, 20]:
    xk = cheb_extrema(n, a_int, b_int)
    yk = y_func(xk)
    y_lag = lagrange_interp(xk, yk, x_fine)
    axes[0].plot(x_fine, y_lag, label=f"n = {n}")

axes[0].plot(x_fine, y_fine, 'k--', lw=2, label="$y(x)$")
axes[0].set_title("Интерполяция Лагранжа на экстремумах Чебышёва")
axes[0].legend(); axes[0].set_xlabel("x"); axes[0].set_ylabel("y")

ns = np.arange(3, 30)
errors = []
for n in ns:
    xk = cheb_extrema(n, a_int, b_int)
    yk = y_func(xk)
    y_lag = lagrange_interp(xk, yk, x_fine)
    errors.append(np.max(np.abs(y_lag - y_fine)))

axes[1].semilogy(ns, errors, 'o-')
axes[1].set_xlabel("n (число узлов Чебышёва)")
axes[1].set_ylabel("max |err|")
axes[1].set_title("Погрешность интерполяции")
axes[1].grid(True)
plt.tight_layout(); plt.show()

### Задача 4  Ряды (1 балл)

Вычислить сумму ряда

$$
\sum_{n=1}^{\infty} \frac{n+1}{n^4+n^2+1} \sin (3x)
$$

с точностью $\varepsilon=10^{-6}$. Обоснуйте, почему исходная точность достигнута.

Ряд $\sum_{n=1}^{\infty} \frac{n+1}{n^4+n^2+1}\sin(3n)$ сходится, т.к. $\frac{n+1}{n^4+n^2+1} \sim \frac{1}{n^3}$.

Применим метод Куммера с вспомогательным рядом $b_n = \frac{\sin(3n)}{n^3}$, сумма которого известна:
$$\sum_{n=1}^{\infty}\frac{\sin(nx)}{n^3} = \frac{\pi^2 x}{6} - \frac{\pi x^2}{4} + \frac{x^3}{12}, \quad x \in [0, 2\pi]$$

При $x = 3$: $S_1 = \frac{3\pi^2}{6} - \frac{9\pi}{4} + \frac{27}{12} = \frac{\pi^2}{2} - \frac{9\pi}{4} + \frac{9}{4}$.

$$\gamma = \lim_{n \to \infty}\frac{a_n}{b_n} = \lim_{n \to \infty}\frac{(n+1)n^3}{n^4+n^2+1} = 1$$

Тогда $S = S_1 + \sum_{n=1}^{\infty}\left(\frac{n+1}{n^4+n^2+1} - \frac{1}{n^3}\right)\sin(3n)$.

Остаток убывает как $O(1/n^5)$, поэтому для $\varepsilon = 10^{-6}$ достаточно порядка 10 членов.

In [ ]:
x_val = 3.0
S1 = np.pi**2 / 2 - 9 * np.pi / 4 + 9.0 / 4

eps = 1e-6

S_corr = 0.0
N_kummer = 0
for n in range(1, 10000):
    a_n = (n + 1) / (n**4 + n**2 + 1)
    b_n = 1.0 / n**3
    term = (a_n - b_n) * np.sin(x_val * n)
    S_corr += term
    if n > 1 and abs(term) < eps:
        N_kummer = n
        break

S_kummer = S1 + S_corr

S_direct = 0.0
N_direct = 0
for n in range(1, 10**7):
    term = (n + 1) / (n**4 + n**2 + 1) * np.sin(x_val * n)
    S_direct += term
    if n > 1 and abs(term) < eps:
        N_direct = n
        break

print(f"Метод Куммера:       S = {S_kummer:.10f},  членов: {N_kummer}")
print(f"Прямое суммирование: S = {S_direct:.10f},  членов: {N_direct}")
print(f"|разница|           = {abs(S_kummer - S_direct):.2e}")
print(f"\nОбоснование: остаточный член убывает как O(1/n^5),")
print(f"при n = {N_kummer}: |term| = {abs((N_kummer+1)/(N_kummer**4+N_kummer**2+1) - 1/N_kummer**3) * 1:.2e} < eps = {eps}")

### Задача 5  Дискретное преобразование Фурье (1.5 балла)

Сгенерируйте два сигнала: сигнал 1 - синусоидальная волна с частотой 5 Гц, амплитудой 3 и фазовым сдвигом 3; сигнал 2 - синусоидальная волна с частотой 2 Гц, амплитудой 2 и фазовым сдвигом -2. Постройте график суммы сигнала в течение 2 секунд. Сделайте дискретное преобразование Фурье. Как из него "вычленить" исходные сигналы? 

In [ ]:
T = 2.0
fs = 1000
t = np.arange(0, T, 1 / fs)

A1, f1, phi1 = 3.0, 5.0, 3.0
A2, f2, phi2 = 2.0, 2.0, -2.0

s1 = A1 * np.sin(2 * np.pi * f1 * t + phi1)
s2 = A2 * np.sin(2 * np.pi * f2 * t + phi2)
signal = s1 + s2

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(t, signal, lw=0.8)
axes[0].set_xlabel("t, с"); axes[0].set_ylabel("Амплитуда")
axes[0].set_title("Суммарный сигнал")

N = len(t)
S = np.fft.fft(signal)
freqs = np.fft.fftfreq(N, 1 / fs)

mask = freqs >= 0
axes[1].stem(freqs[mask], 2 * np.abs(S[mask]) / N, markerfmt='o', basefmt=' ')
axes[1].set_xlim(0, 15)
axes[1].set_xlabel("Частота, Гц"); axes[1].set_ylabel("Амплитуда")
axes[1].set_title("Спектр (ДПФ)")
plt.tight_layout(); plt.show()

idx_pos = freqs > 0
peak_freqs = freqs[idx_pos]
peak_amps = 2 * np.abs(S[idx_pos]) / N
peak_phases = np.angle(S[idx_pos])

top2 = np.argsort(peak_amps)[-2:]
print("Восстановленные параметры сигналов:")
for i, idx in enumerate(sorted(top2)):
    amp = peak_amps[idx]
    freq = peak_freqs[idx]
    phase = peak_phases[idx] + np.pi / 2  # sin = cos с фазовым сдвигом -π/2
    phase = (phase + np.pi) % (2 * np.pi) - np.pi
    print(f"  Сигнал {i+1}: A = {amp:.2f}, f = {freq:.1f} Гц, φ = {phase:.2f} рад")
print(f"\nИсходные:  A1={A1}, f1={f1} Гц, φ1={phi1};  A2={A2}, f2={f2} Гц, φ2={phi2}")